# Lab L5.3 · Capstone — ship, govern and monitor the MIZAN churn-retention service
**Horizon Gulf Bank · Project MIZAN** · CAIP AIP-210 · Certified AI Practitioner · Day 5 · ⏱ 90 min · **Level 2 · Guided starter (fill in the blanks) · ★ recommended level**

Take the Day 3 churn champion all the way: integrity hash, fairness and explainability gate, MLflow registry, a secured API with reason codes and an audit trail, a campaign batch job, a drift simulation and a rollback drill.

### How this notebook works
- Every `✏️ BLANK n` marks code **you** write. Inline blanks look like `____`; block blanks say `# ...your code here...`.
- The markdown cell above each code cell explains **what** to write, **where**, and **why**.
- Stuck > 5 min? Open the 💡 hint in the HTML lab guide, then ask your trainer. The Level 3 solution is released after the lab.

### Scenario
**Horizon Gulf Bank (HGB)** is a fictional UAE retail bank with 250,000 customers. Its Board approved **Project MIZAN** — an AI decisioning platform covering credit, fraud, collateral valuation, cash logistics, customer retention and document automation. You are a junior ML practitioner in the MIZAN squad.

**Capstone brief.** The Retention Committee approved the churn programme on one condition: the model must go live *the way a regulated bank deploys models*. Working in teams of 3–4, you will package the champion, prove it is fair and explainable, register it, serve it securely, generate the first campaign list, rehearse what happens when the population drifts and when a bad model version is pushed — and present the result to the committee (5 minutes per team).

**Progressive architecture:** this lab consumes artefacts from all four previous days and uses the same MLflow database as L5.1.

In [ ]:
# ── Project MIZAN course setup ─ run this cell first ──────────────────────────
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the course root (the folder that contains data/generate_data.py) from wherever this notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "generate_data.py").exists())
DATA = ROOT / "data"          # raw synthetic datasets (read-only)
CKPT = ROOT / "checkpoints"   # hand-off artefacts between labs (lets you start any lab fresh)
CKPT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
RNG = 42
print("Course root:", ROOT)

## Step 1 · Package the champion and prove its integrity

**Why this matters.** Before anything is deployed, Model Risk must be sure the artefact is *exactly* the one that was validated. A **SHA-256 hash** of the file is its fingerprint: any change — retraining, tampering, corruption — changes the hash. We re-score the test set to prove the packaged model reproduces the validated metric.

#### ✏️ What to write in the next cell
- **Blank 1** *(inline)* — top 10% of training scores = HIGH
- **Blank 2** *(inline)* — fingerprint of the exact bytes

In [ ]:
import json
cust = pd.read_parquet(CKPT / "customers_clean.parquet")                    # L1.1
seg = pd.read_parquet(CKPT / "customer_segments.parquet")                  # L3.2
txf = pd.read_parquet(CKPT / "customer_txn_features.parquet")              # L1.3
TXN = ["has_card_activity", "txn_count", "log_amt_mean", "recency_days", "spend_trend", "ecom_share", "n_mcc"]
df = (cust.merge(seg[["customer_id", "segment"]], on="customer_id", validate="one_to_one")
          .merge(txf[["customer_id"] + TXN], on="customer_id", validate="one_to_one"))
df["log_balance"], df["log_salary"] = np.log1p(df.avg_balance_aed), np.log1p(df.monthly_salary_aed)
NUM = ["age", "tenure_months", "num_products", "salary_transfer", "has_credit_card", "has_personal_loan", "has_auto_loan", "has_mortgage",
       "log_balance", "log_salary", "digital_logins_30d", "branch_visits_90d", "complaints_12m", "days_since_last_txn", "aecb_score"] + TXN
CAT = ["segment", "employment_type", "emirate"]
FORBIDDEN = {"account_closed_date", "retention_offer_made", "gender", "nationality_group"}   # leakage + protected attributes
import joblib, hashlib, json, shutil, sys, importlib
from sklearn.metrics import roc_auc_score, average_precision_score
SVC = CKPT / "mizan_churn"; (SVC / "model").mkdir(parents=True, exist_ok=True)
champ = joblib.load(CKPT / "churn_model.joblib")
ids = champ["split_ids"]
tr, va, te = (df[df.customer_id.isin(ids[s])] for s in ["train", "valid", "test"])
FEATS, model = champ["features"], champ["model"]
p_tr = model.predict_proba(tr[FEATS])[:, 1]
bundle = {"model": model, "features": FEATS, "name": "mizan-churn",
          "reference": {**tr[NUM].median().to_dict(), **{c: tr[c].mode()[0] for c in CAT}},
          "bands": {"medium": float(np.quantile(p_tr, 0.80)), "high": float(____)}}  # ✏️ B1
joblib.dump(bundle, SVC / "model" / "churn_model.joblib")
def sha256(path):
    return ____  # ✏️ B2
digest = sha256(SVC / "model" / "churn_model.joblib")
p_te = model.predict_proba(te[FEATS])[:, 1]
pr_te = average_precision_score(te.churned_6m, p_te)
assert abs(pr_te - champ["test"]["PR_AUC"]) < 1e-9, "packaged model does not reproduce the validated metric!"
print(f"{champ['name']} packaged | sha256 {digest[:16]}… | test PR-AUC reproduced: {pr_te:.4f} | bands {bundle['bands']}")

**✔ Checkpoint.** Logistic champion packaged; the hash is printed; test PR-AUC 0.2514 reproduced exactly; MEDIUM starts at P(churn) ≈ 0.18, HIGH at ≈ 0.24.

## Step 2 · Responsible-AI gate: fairness and reason codes

**Why this matters.** Gender and nationality are **not** model inputs — but features can act as **proxies**. The gate measures outcomes: the **disparate impact ratio** (each group's rate of being *selected for the HIGH band* ÷ the highest group's rate; the 4/5 rule flags < 0.8) and **equal opportunity** (true-positive rate among actual churners per group). Being selected for a retention offer is a *benefit*, so under-selection is the harm to look for. Then every decision gets **reason codes**.

#### ✏️ What to write in the next cell
- **Blank 3** *(inline)* — each group vs the most-selected group
- **Blank 4** *(inline)* — probability drop when this feature looks ‘typical’

In [ ]:
T = te.assign(p=p_te, high=p_te >= bundle["bands"]["high"],
              age_band=pd.cut(te.age, [17, 29, 44, 59, 120], labels=["18-29", "30-44", "45-59", "60+"]))
def fairness(by):
    g = T.groupby(by, observed=True)
    out = pd.DataFrame({"customers": g.size(), "selection_rate": g.high.mean(),
                        "TPR": T[T.churned_6m == 1].groupby(by, observed=True).high.mean()})   # share of actual churners selected
    out["DI_ratio"] = ____  # ✏️ B3
    out["EO_ratio"] = out.TPR / out.TPR.max()                          # equal opportunity: churners reached vs best-served group
    out["flag"] = np.where(out.EO_ratio < 0.7, "BLOCK", np.where(out.DI_ratio < 0.8, "REVIEW", "ok"))
    return out
fair = {by: fairness(by) for by in ["gender", "nationality_group", "age_band"]}
for by, t in fair.items(): print(f"— {by}"); display(t.round(3))
min_di = min(t.DI_ratio.min() for t in fair.values())
REF = bundle["reference"]
py = lambda v: v.item() if hasattr(v, "item") else v                 # numpy scalar → plain Python
def reason_codes(row: pd.DataFrame, k=3):
    p = model.predict_proba(row[FEATS])[:, 1][0]
    what_if = pd.concat([row[FEATS]] * len(FEATS), ignore_index=True).astype({f: float for f in NUM})   # floats accept median values
    for i, f in enumerate(FEATS): what_if.loc[i, f] = REF[f]
    delta = ____  # ✏️ B4
    top = np.argsort(-delta)[:k]
    return [(FEATS[i], py(row[FEATS[i]].iloc[0]), f"+{delta[i] * 100:.1f} pts") for i in top if delta[i] > 0]
for cid in T.nlargest(3, "p").customer_id:
    r = T[T.customer_id == cid]; print(cid, f"P={r.p.iloc[0]:.2f}", reason_codes(r))

**✔ Checkpoint.** Gender is balanced (DI ≈ 0.99). **Emirati (DI ≈ 0.61) and Western (≈ 0.67) customers are flagged REVIEW** — they are selected less often, and the 60+ band (small, 651 customers) is selected most. Equal-opportunity ratios stay ≥ 0.7 for gender and nationality (lowest ≈ 0.74, Emirati), so nothing is BLOCKed — but it is close. Reason codes for the riskiest customers cite complaints, no salary transfer and inactivity, with the percentage points each adds.

## Step 3 · Register behind an automated release gate

**Why this matters.** The registry is the single source of truth for *which model is live*. A **release gate** — code, not a meeting — checks minimum performance, fairness, forbidden features and integrity before the `champion` alias can move. Every check and its evidence is stored, so auditors can replay the decision.

#### ✏️ What to write in the next cell
- **Blank 5** *(inline)* — every check must pass
- **Blank 6** *(inline)* — the alias is what production resolves

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient
MLDIR = CKPT / "mlflow"; MLDIR.mkdir(exist_ok=True)
mlflow.set_tracking_uri("sqlite:///" + (MLDIR / "mlflow.db").as_posix())
exp = mlflow.get_experiment_by_name("mizan-churn") or mlflow.get_experiment(mlflow.create_experiment("mizan-churn", artifact_location=(MLDIR / "artifacts_churn").as_uri()))
mlflow.set_experiment(experiment_id=exp.experiment_id)
top = np.argsort(-p_te)[: len(p_te) // 10]
metrics = {"test_PR_AUC": pr_te, "test_ROC_AUC": roc_auc_score(te.churned_6m, p_te),
           "test_capture_at_10pct": te.churned_6m.to_numpy()[top].sum() / te.churned_6m.sum(), "min_DI_ratio": min_di}
card = {"name": "mizan-churn", "version": "1.0.0", "algorithm": champ["name"], "artefact_sha256": digest,
        "intended_use": "Rank retail customers for proactive retention calls (≈ 10% of base per half-year). NOT for pricing or credit decisions.",
        "training_data": "customers_clean + transaction features + segments, snapshot 2025-10-01; stratified 60/20/20", "features": FEATS,
        "metrics": metrics, "fairness": {by: t.DI_ratio.round(3).to_dict() for by, t in fair.items()},
        "limitations": ["Probabilities assume the 12% base rate",
                        "Selection-rate disparities (DI < 0.8) for some nationality groups and age bands — they track lower churn rates; under review",
                        "Not validated for SME or private-banking customers"],
        "owners": {"business": "Head of Retail Retention", "model": "MIZAN squad", "validation": "Model Risk Management"}}
(SVC / "model" / "model_card.json").write_text(json.dumps(card, indent=2, default=str))
def release_gate(m, feats, digest_now, digest_packaged=digest):
    checks = {"PR_AUC >= 0.22": m["test_PR_AUC"] >= 0.22, "capture@10% >= 0.22": m["test_capture_at_10pct"] >= 0.22,
              "no forbidden features": not FORBIDDEN & set(feats),
              "gender & nationality equal-opportunity ratio >= 0.7": all(fair[b].EO_ratio.min() >= 0.7 for b in ["gender", "nationality_group"]),
              "artefact unchanged since packaging (sha256)": digest_now == digest_packaged}
    checks = {k: bool(v) for k, v in checks.items()}
    return {"checks": checks, "passed": ____}  # ✏️ B5
gate_v1 = release_gate(metrics, FEATS, sha256(SVC / "model" / "churn_model.joblib"))
with mlflow.start_run(run_name="churn-v1.0.0") as run:
    mlflow.log_params({"algorithm": champ["name"], "n_features": len(FEATS)}); mlflow.log_metrics({k: float(v) for k, v in metrics.items()})
    mlflow.log_artifacts(str(SVC / "model"), artifact_path="model"); mlflow.set_tag("gate_passed", gate_v1["passed"])
mc = MlflowClient()
if not mc.search_registered_models(filter_string="name='mizan-churn'"):
    mc.create_registered_model("mizan-churn", description="MIZAN churn-retention ranking model")
mv1 = mc.create_model_version("mizan-churn", f"{run.info.artifact_uri}/model", run.info.run_id)
if gate_v1["passed"]:
    ____  # ✏️ B6
print(json.dumps(gate_v1, indent=1)); print(f"mizan-churn v{mv1.version} registered; champion → v{mc.get_model_version_by_alias('mizan-churn', 'champion').version}")

**✔ Checkpoint.** All five checks pass (DI flags are documented in the card's limitations but are not blocking); version registered and aliased `champion`.

## Step 4 · Serve it securely: validation, API key, reason codes, audit log

**Why this matters.** The CRM calls the service, not the notebook. **Pydantic** rejects malformed or out-of-range input; an **API key** header is the minimum access control (production adds mTLS/OAuth and a secrets manager); every decision is written to an **audit log** with a request id and model version; high-value, high-risk customers are routed to a **human** relationship manager.

In [ ]:
%%writefile ../../../checkpoints/mizan_churn/app.py
"""MIZAN churn-retention scoring service (capstone L5.3)."""
import json, os, time, uuid
from pathlib import Path
from typing import List, Literal, Optional

import joblib
import numpy as np
import pandas as pd
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field

HERE = Path(__file__).parent
BUNDLE = joblib.load(HERE / "model" / "churn_model.joblib")
CARD = json.loads((HERE / "model" / "model_card.json").read_text())
MODEL, FEATS, REF, BANDS = BUNDLE["model"], BUNDLE["features"], BUNDLE["reference"], BUNDLE["bands"]
API_KEY = os.environ.get("MIZAN_API_KEY", "dev-only-key")          # real deployments read this from a secret store
AUDIT = HERE / "audit_log.jsonl"
app = FastAPI(title="MIZAN churn service", version=CARD["version"])

EMIRATES = Literal["Abu Dhabi", "Ajman", "Dubai", "Fujairah", "Ras Al Khaimah", "Sharjah", "Umm Al Quwain"]

class Customer(BaseModel):
    customer_id: str = Field(pattern=r"^HGB\d{7}$")
    age: int = Field(ge=18, le=100)
    tenure_months: int = Field(ge=0, le=600)
    num_products: int = Field(ge=0, le=15)
    salary_transfer: int = Field(ge=0, le=1)
    has_credit_card: int = Field(ge=0, le=1)
    has_personal_loan: int = Field(ge=0, le=1)
    has_auto_loan: int = Field(ge=0, le=1)
    has_mortgage: int = Field(ge=0, le=1)
    avg_balance_aed: float = Field(ge=0, le=1e9)
    monthly_salary_aed: float = Field(ge=0, le=1e7)
    digital_logins_30d: int = Field(ge=0, le=1000)
    branch_visits_90d: int = Field(ge=0, le=500)
    complaints_12m: int = Field(ge=0, le=100)
    days_since_last_txn: int = Field(ge=0, le=5000)
    aecb_score: Optional[float] = Field(None, ge=300, le=900)
    has_card_activity: int = Field(ge=0, le=1)
    txn_count: float = Field(ge=0)
    log_amt_mean: float = Field(ge=0, le=20)
    recency_days: float = Field(ge=0, le=999)
    spend_trend: float = Field(ge=-30, le=30)
    ecom_share: float = Field(ge=0, le=1)
    n_mcc: float = Field(ge=0, le=50)
    segment: int = Field(ge=0, le=20)
    employment_type: Literal["Private", "Government", "Self_Employed", "Retired"]
    emirate: EMIRATES

def _frame(customers: List[Customer]) -> pd.DataFrame:
    d = pd.DataFrame([c.model_dump() for c in customers])
    d["log_balance"], d["log_salary"] = np.log1p(d.avg_balance_aed), np.log1p(d.monthly_salary_aed)   # same transform as training
    d["aecb_score"] = d.aecb_score.astype(float)
    return d

def reasons(row: pd.DataFrame, p: float, k: int = 3) -> list:
    """Model-agnostic reason codes: how much does P(churn) fall if one feature is set to a typical value?"""
    what_if = pd.concat([row[FEATS]] * len(FEATS), ignore_index=True)
    what_if = what_if.astype({f: float for f in FEATS if f != "segment" and pd.api.types.is_numeric_dtype(what_if[f])})
    for i, f in enumerate(FEATS):
        what_if.loc[i, f] = REF[f]
    delta = p - MODEL.predict_proba(what_if)[:, 1]
    top = np.argsort(-delta)[:k]
    py = lambda v: v.item() if hasattr(v, "item") else v                  # numpy scalar → JSON-friendly Python value
    return [{"feature": FEATS[i], "value": py(row[FEATS[i]].iloc[0]), "adds_pct_points": round(float(delta[i]) * 100, 1)} for i in top if delta[i] > 0]

def band(p: float) -> str:
    return "HIGH" if p >= BANDS["high"] else ("MEDIUM" if p >= BANDS["medium"] else "LOW")

def _audit(event: dict):
    with AUDIT.open("a", encoding="utf-8") as f:
        f.write(json.dumps(event, default=str) + "\n")

def _check_key(key: Optional[str]):
    if key != API_KEY:
        raise HTTPException(status_code=401, detail="invalid or missing API key")

@app.get("/health")
def health():
    return {"status": "ok", "model": CARD["name"], "version": CARD["version"], "sha256": CARD["artefact_sha256"][:12]}

@app.post("/score")
def score(c: Customer, x_api_key: Optional[str] = Header(None)):
    _check_key(x_api_key)
    t0 = time.perf_counter()
    row = _frame([c])
    p = float(MODEL.predict_proba(row[FEATS])[:, 1][0])
    out = {"request_id": str(uuid.uuid4()), "customer_id": c.customer_id, "p_churn": round(p, 4), "risk_band": band(p),
           "reasons": reasons(row, p), "model_version": CARD["version"],
           "human_review": band(p) == "HIGH" and c.avg_balance_aed > 500_000}      # high-value, high-risk → relationship manager decides
    out["latency_ms"] = round((time.perf_counter() - t0) * 1000, 2)
    _audit({"ts": time.time(), **{k: out[k] for k in ["request_id", "customer_id", "p_churn", "risk_band", "model_version"]}})
    return out

#### ✏️ What to write in the next cell
- **Blank 7** *(inline)* — the caller proves who it is

In [ ]:
sys.path.insert(0, str(CKPT))
import mizan_churn.app as svc; importlib.reload(svc)
from fastapi.testclient import TestClient
client = TestClient(svc.app)
print(client.get("/health").json())
RAW = [f for f in svc.Customer.model_fields]
def payload(row: pd.Series) -> dict:
    return {k: (None if pd.isna(row[k]) else (row[k].item() if hasattr(row[k], "item") else row[k])) for k in RAW}
risky = T.nlargest(1, "p").iloc[0]
ok = client.post("/score", json=payload(risky), headers={____})  # ✏️ B7
print(ok.status_code, json.dumps(ok.json(), indent=1, default=str))
print("no key  →", client.post("/score", json=payload(risky)).status_code)
bad = client.post("/score", json=payload(risky) | {"emirate": "Atlantis", "age": 12}, headers={"x-api-key": "dev-only-key"})
print("invalid →", bad.status_code, [e["loc"][-1] for e in bad.json()["detail"]])
print("audit   →", (SVC / "audit_log.jsonl").read_text().strip().splitlines()[-1])

**✔ Checkpoint.** `/health` shows version 1.0.0 and the hash prefix. The riskiest customer scores 0.76 (HIGH) with 3 reason codes (complaints +21 pts, no salary transfer +19 pts, 31 idle days +13 pts) in ≈ 10 ms; no key → **401**; invalid emirate and age → **422** naming both fields; the audit line has request id, score and model version.

## Step 5 · Batch job: the first retention campaign

**Why this matters.** Real-time scoring serves the call-centre screen; the campaign itself is a **batch** job over the whole base. We score all customers, rank by **expected value** (L3.4), keep the 10% budget, attach reason codes and write the file the CRM imports.

#### ✏️ What to write in the next cell
- **Blank 8** *(inline)* — 10% call budget, highest expected value first

In [ ]:
t0 = time.time()
allp = df.assign(p=model.predict_proba(df[FEATS])[:, 1])
allp["value"] = 0.02 * allp.avg_balance_aed + 400
allp["ev"] = allp.p * 0.30 * allp.value - (25 + allp.p * 0.30 * 150)
camp = ____.copy()  # ✏️ B8
camp["band"] = np.where(camp.p >= bundle["bands"]["high"], "HIGH", np.where(camp.p >= bundle["bands"]["medium"], "MEDIUM", "LOW"))
camp["reasons"] = "batch-later"
for i in camp.index[:2000]:
    camp.at[i, "reasons"] = "; ".join(f"{f}={v}" for f, v, _ in reason_codes(camp.loc[[i]], k=2))
camp[["customer_id", "p", "ev", "band", "reasons"]].to_csv(CKPT / "retention_campaign.csv", index=False)
print(f"{len(camp):,} customers | expected value AED {camp.ev.sum():,.0f} | {time.time() - t0:.0f}s"); display(camp.band.value_counts())
camp[["customer_id", "p", "ev", "band", "reasons"]].head(5).round(3)

**✔ Checkpoint.** 25,000 customers with ≈ AED 1.62M expected value. The list is **not** only HIGH-band (≈ 8.1k HIGH, 4.9k MEDIUM, 12.0k LOW): EV ranking pulls in wealthy lower-risk customers. Reasons read like ‘complaints_12m=3; salary_transfer=0’.

## Step 6 · Next quarter: simulate drift and decide

**Why this matters.** Q1 2026 brings a mobile-app outage (logins fall), a complaints spike and a competitor's salary-transfer campaign. **PSI** per feature quantifies the data drift; PSI on the scores quantifies **prediction drift**. Without labels yet (churn is only known 6 months later) these are the early-warning signals that trigger investigation or retraining.

#### ✏️ What to write in the next cell
- **Blank 9** *(inline)* — Population Stability Index

In [ ]:
rng = np.random.default_rng(7)
q1 = df.sample(50_000, random_state=RNG).copy()
q1["digital_logins_30d"] = (q1.digital_logins_30d * 0.6).round()
q1.loc[rng.random(len(q1)) < 0.15, "complaints_12m"] += 1
q1["salary_transfer"] = np.where((q1.salary_transfer == 1) & (rng.random(len(q1)) < 0.10), 0, q1.salary_transfer)
def psi(ref, new, bins=10):
    ref, new = np.asarray(ref, float), np.asarray(new, float)
    ref, new = ref[~np.isnan(ref)], new[~np.isnan(new)]
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3: edges = np.unique(np.r_[ref.min() - 1, np.unique(ref), ref.max() + 1])
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.histogram(ref, edges)[0] / len(ref) + 1e-6
    n = np.histogram(new, edges)[0] / len(new) + 1e-6
    return float(____)  # ✏️ B9
drift = pd.Series({f: psi(tr[f], q1[f]) for f in NUM}).sort_values(ascending=False)
q1_p = model.predict_proba(q1[FEATS])[:, 1]
psi_score = psi(p_tr, q1_p)
display(drift.head(6).round(3).to_frame("PSI"))
status = "RETRAIN TRIGGER" if max(drift.max(), psi_score) > 0.25 else ("INVESTIGATE" if max(drift.max(), psi_score) > 0.1 else "STABLE")
print(f"score PSI {psi_score:.3f} | mean P(churn) {p_tr.mean():.3f} → {q1_p.mean():.3f} | HIGH band share "
      f"{(p_tr >= bundle['bands']['high']).mean():.1%} → {(q1_p >= bundle['bands']['high']).mean():.1%} | decision: {status}")

**✔ Checkpoint.** `digital_logins_30d` has an enormous PSI (≈ 4.7 — the whole distribution moved), complaints ≈ 0.09, salary transfer ≈ 0.02; the score PSI is ≈ 0.22, mean P(churn) rises 0.12 → 0.17 and the HIGH band more than doubles (10% → ≈ 22%). Decision: **RETRAIN TRIGGER** — but first check whether the login drop is a data/outage artefact rather than real behaviour.

## Step 7 · Rollback drill and the committee summary

**Why this matters.** Someone will eventually push a broken model. Here a broken join silently **zero-fills three features** (salary transfer, complaints, days since last transaction). The gate must catch it, the `champion` alias must stay on v1, and the team must know the one-line **rollback**. Then you assemble the numbers for the 5-minute committee presentation.

#### ✏️ What to write in the next cell
- **Blank 10** *(inline)* — a different artefact has a different fingerprint

In [ ]:
from sklearn.base import clone
bug = lambda d: d.assign(salary_transfer=0, complaints_12m=0, days_since_last_txn=0)   # a broken join silently zero-fills 3 columns
v2 = clone(model).fit(bug(tr)[FEATS], tr.churned_6m)
p2 = v2.predict_proba(bug(te)[FEATS])[:, 1]
top2 = np.argsort(-p2)[: len(p2) // 10]
m2 = {"test_PR_AUC": average_precision_score(te.churned_6m, p2), "test_capture_at_10pct": te.churned_6m.to_numpy()[top2].sum() / te.churned_6m.sum()}
(SVC / "candidate_v2").mkdir(exist_ok=True); joblib.dump({**bundle, "model": v2}, SVC / "candidate_v2" / "churn_model.joblib")
digest_v2 = ____  # ✏️ B10
gate_v2 = release_gate(m2, FEATS, digest_v2, digest_packaged=digest_v2)
with mlflow.start_run(run_name="churn-v2-candidate") as run2:
    mlflow.log_metrics({k: float(v) for k, v in m2.items()}); mlflow.set_tag("gate_passed", gate_v2["passed"])
mv2 = mc.create_model_version("mizan-churn", f"{run2.info.artifact_uri}/model", run2.info.run_id)
if gate_v2["passed"]: mc.set_registered_model_alias("mizan-churn", "champion", mv2.version)
live = mc.get_model_version_by_alias("mizan-churn", "champion").version
print("candidate gate:", gate_v2["checks"], "→ passed" if gate_v2["passed"] else "→ REJECTED",
      f"| registry: v1.0.0 = version {mv1.version}, candidate = version {mv2.version}, champion alias → version {live}")
print(f"rollback one-liner if needed: mc.set_registered_model_alias('mizan-churn', 'champion', '{mv1.version}')")
(CKPT / "churn_release_gate.json").write_text(json.dumps({"v1": {"version": mv1.version, **gate_v1, "metrics": metrics},
    "v2": {"version": mv2.version, **gate_v2, "metrics": m2}, "champion_alias": live}, indent=2, default=float))
nn = json.loads((CKPT / "nn_tuning_results.json").read_text()) if (CKPT / "nn_tuning_results.json").exists() else {"decision": "n/a"}
summary = f"""# MIZAN churn-retention service — committee summary
* **Model:** {champ['name']} v1.0.0 · sha256 `{digest[:16]}…` · MLflow `mizan-churn@champion` = v{live}
* **Test performance:** PR-AUC {pr_te:.3f} · ROC-AUC {metrics['test_ROC_AUC']:.3f} · top-10% capture {metrics['test_capture_at_10pct']:.1%} (random = 10%)
* **Challenger (L4.3 neural network):** {nn['decision']}
* **Fairness:** min equal-opportunity ratio gender/nationality {min(fair[b].EO_ratio.min() for b in ['gender', 'nationality_group']):.2f} (gate ≥ 0.70) · min DI {min_di:.2f} (REVIEW flags documented)
* **Campaign:** {len(camp):,} customers · expected value AED {camp.ev.sum():,.0f}
* **Q1 drift simulation:** score PSI {psi_score:.3f} · decision {status}
* **Release gate:** v1 passed · v2 (feature bug) rejected — PR-AUC {m2['test_PR_AUC']:.3f}
"""
(CKPT / "capstone_summary.md").write_text(summary, encoding="utf-8"); print(summary)

**✔ Checkpoint.** The candidate's PR-AUC drops to ≈ 0.21 and capture@10% below 0.22 → both performance checks fail → **REJECTED**; the `champion` alias stays on the v1.0.0 version. (Registry version numbers grow each time you re-run — that is the point of a registry.) The summary prints the headline numbers for the presentation.

## ✅ Lab complete
🎓 **Capstone complete.** You have taken one model from raw data (Day 1) to a governed, monitored production service (Day 5). Prepare your team's 5-minute committee presentation from `capstone_summary.md`.